# 17 — From RNN to LSTM: what the hidden state forgets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/017-rnn-to-lstm.ipynb)

A recurrent network carries one vector of memory from step to step and mixes in each new input. In this notebook you run a single tanh unit for twenty steps, plot how its memory fades, and drag the recurrent weight to see the difference between a healthy and a dying signal. The last section shows why the LSTM was invented.

## 1. The numbers: one pulse, twenty steps

One number is both the input and the memory here, so nothing hides. The input is a single pulse at step zero followed by silence, which isolates the memory term. `W` is the recurrent weight: the fraction of the old state that is offered back to the unit.

In [ ]:
import numpy as np                              # the array library, preinstalled

W = 0.9                                         # how much of the old state is passed forward
x = np.zeros(20)                                # twenty time steps of silence
x[0] = 1.0                                      # except step 0: one pulse goes in
h = 0.0                                         # the hidden state starts empty
states = []                                     # every h_t, kept for plotting

for t in range(20):                             # one step per time index
    h = np.tanh(W * h + x[t])                   # the RNN rule: memory plus new input, squashed
    states.append(h)                            # record the state

print("h at step 0:", round(states[0], 3))      # 0.762: nearly the whole pulse
print("h at step 9:", round(states[9], 3))      # 0.189: about a quarter left
print("h at step 19:", round(states[19], 3))    # 0.063: eight percent of the pulse

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>W = 0.9</code> — the recurrent weight, the memory's leak rate.</li><li><code>x = np.zeros(20)</code> — a sequence of twenty inputs, all zero.</li><li><code>x[0] = 1.0</code> — one pulse at the first step.</li><li><code>np.tanh(...)</code> — squashes any number into -1 to 1.</li><li><code>h = np.tanh(W * h + x[t])</code> — the whole RNN rule on one line.</li><li><code>states.append(h)</code> — save each state so the plot can use it.</li></ul>

</details>

## 2. Step by step: watch the memory term shrink

Print the three pieces separately: the new input, the memory it meets, and the state that comes out. The memory term `W * h` is what the past contributes, and it is strictly smaller than the state before it. That gap is the forgetting.

In [ ]:
h = 0.0                                         # reset the state
W = 0.9                                         # the same recurrent weight
print(f"{'t':>2} {'x_t':>6} {'W*h':>7} {'h_t':>7}")   # a header row
for t in range(20):                             # one printed line per step
    memory = W * h                              # what the past offers to this step
    h = np.tanh(memory + x[t])                  # add the input, squash, that is the new state
    print(f"{t:>2} {x[t]:>6.2f} {memory:>7.4f} {h:>7.4f}")   # input, memory, state
    if t == 0:                                  # only the first row is interesting
        print("  first step: memory was 0, so h = tanh(1.0)")   # explain the start

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>h = 0.0</code> — start again from an empty state.</li><li><code>print(f"{'t':&gt;2} ...")</code> — a formatted header, right-aligned columns.</li><li><code>memory = W * h</code> — the past's contribution before the new input.</li><li><code>h = np.tanh(memory + x[t])</code> — same rule as section 1, written out.</li><li><code>if t == 0</code> — annotate only the first row.</li></ul>

</details>

## 3. Plot: the state and the retained signal

The dots are the real states; the dashed line is the same decay with `tanh` removed, which is close to a pure `0.9^t`. Both say the same thing: information leaves geometrically, so twenty steps is already far too long.

In [ ]:
import matplotlib.pyplot as plt                 # plotting library, preinstalled

t = np.arange(20)                               # the time indices
linear = W ** t                                 # how much survives if tanh were linear
plt.figure(figsize=(6, 3))                      # wide, short figure
plt.plot(t, states, marker="o", markersize=3, color="tab:blue", label="h_t (real, with tanh)")   # actual state
plt.plot(t, linear, linestyle="--", color="tab:red", label="W^t (linear memory)")                # pure decay
plt.xlabel("time step")                         # x-axis label
plt.ylabel("size of the state")                 # y-axis label
plt.title("0.76 at the start, 0.06 after twenty steps")   # the takeaway
plt.legend()                                    # name both curves
plt.grid(alpha=0.3)                             # faint grid for reading values
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>t = np.arange(20)</code> — the x values, one per step.</li><li><code>linear = W ** t</code> — element-wise power, the decay without squashing.</li><li><code>plt.plot(t, states, ...)</code> — the measured states as dots joined by a line.</li><li><code>plt.plot(t, linear, ...)</code> — the reference decay as a dashed line.</li><li><code>plt.xlabel</code> / <code>plt.ylabel</code> / <code>plt.title</code> — axes and the takeaway.</li></ul>

</details>

## 4. Interactive: turn the leak rate

Now the slider sets `W`. Small values kill the memory within a few steps, `W` near 1 keeps it alive, and anything above 1 makes the state saturate at the ceiling of `tanh`. The title reports the state after twenty steps.

In [ ]:
from ipywidgets import interact                   # sliders that rerun a function

def run(W=0.9):                                 # W comes from the slider
    h = 0.0                                     # empty state
    hist = []                                   # one entry per step
    for t in range(20):                         # twenty steps
        h = np.tanh(W * h + x[t])               # the RNN rule again
        hist.append(h)                          # save it
    plt.figure(figsize=(6, 3))                  # fresh figure per drag
    plt.plot(range(20), hist, marker="o", markersize=3, color="tab:blue")   # the real run
    plt.plot(range(20), W ** np.arange(20), linestyle="--", color="grey", label="W^t")   # reference
    plt.ylim(0, 1.05)                           # tanh output cannot exceed 1
    plt.xlabel("time step")                     # x-axis label
    plt.ylabel("size of the state")             # y-axis label
    plt.title(f"W = {W:.2f} -> h after 20 steps = {hist[-1]:.4f}")   # live readout
    plt.legend()                                # name the reference curve
    plt.show()                                  # draw the frame

interact(run, W=(0.0, 1.2, 0.05))               # slider: 0 to 1.2, steps of 0.05

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>interact</code> — attaches a slider to <code>run</code>.</li><li><code>def run(W=0.9)</code> — one frame per slider value.</li><li><code>hist = []</code> — collects the twenty states for this run.</li><li><code>plt.ylim(0, 1.05)</code> — fixed y range so curves do not jump.</li><li><code>f"W = {W:.2f} ... {hist[-1]:.4f}"</code> — the readout: weight and final state.</li><li><code>interact(run, W=(0.0, 1.2, 0.05))</code> — range and step.</li></ul>

</details>

## 5. When it breaks: the gradient decays as 0.9^20

Training sends an error signal backwards through the same multiplication, so the gradient shrinks by `W` at every step it travels. After twenty steps only about an eighth of the signal is left, and after fifty it is a rounding error. This is why a plain RNN cannot learn long-range dependencies.

In [ ]:
ks = np.arange(1, 51)                           # steps travelled backwards
factor = 0.9 ** ks                              # the gradient multiplier at each distance
print("20 steps back:", f"{0.9 ** 20:.4f}")     # 0.1216 of the original signal
print("50 steps back:", f"{0.9 ** 50:.2e}")     # 0.005: effectively zero
for L in [5, 10, 20, 40]:                       # a few distances, spelled out
    print(f"distance {L:>2}: gradient x {0.9 ** L:.4f}")   # the shrinking factor

plt.figure(figsize=(6, 3))                      # wide, short figure
plt.semilogy(ks, factor, color="tab:red")        # log scale, otherwise the curve is invisible
plt.axhline(1e-3, color="grey", linestyle=":", label="one thousandth")   # the floor that stops learning
plt.axvline(20, color="tab:blue", linestyle="--", label="20 steps")      # the case from section 1
plt.xlabel("steps between cause and effect")    # x-axis label
plt.ylabel("gradient factor (log scale)")       # y-axis label
plt.title("Long-range learning dies at 0.9 per step")   # the takeaway
plt.legend()                                    # name the two guide lines
plt.show()                                      # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>ks = np.arange(1, 51)</code> — distances from 1 to 50 steps.</li><li><code>factor = 0.9 ** ks</code> — the multiplier applied once per step.</li><li><code>f"{0.9 ** 20:.4f}"</code> — the twenty-step case, about 0.12.</li><li><code>plt.semilogy(...)</code> — log y-axis, the only way to see 1e-5.</li><li><code>plt.axhline(1e-3, ...)</code> — mark where the signal stops mattering.</li><li><code>plt.axvline(20, ...)</code> — mark the twenty-step case.</li></ul>

</details>

## Your turn

Set the section 4 slider to 0.5 and estimate how many steps pass before the state drops under 0.01. Then set it to 1.0 and explain why the state stops shrinking — compare both with the 0.9-per-step decay in section 5.